Final Feature Names:

Train set: train_model_df

Validation set: val_model_df

Test set: test_model_df

Tier 1 Feature Names:
- article_age_hours: Hours since the article was first observed in the dataset
- prior_exposures: Number of times the article was previously shown
- prior_clicks: Number of previous clicks on the article
- prior_ctr: Historical click-through rate of the article
- behavior_confidence: Confidence in the article's own CTR based on how many exposures it has received
- recent_exposures_6h: Article exposures in the previous 6 hours
- recent_clicks_6h: Article clicks in the previous 6 hours
- recent_ctr_6h: Article CTR in the previous 6 hours
- ctr_momentum_6h: Change in article CTR between the most recent 6-hour window and the preceding 6-hour window

Tier 2 Feature Names:
- category_recent_ctr_6h: Recent CTR of other articles in the same category
- category_ctr_momentum_6h: Change in CTR of other articles in the same category
- subcategory_recent_ctr_6h: Recent CTR of other articles in the same subcategory
- subcategory_ctr_momentum_6h: Change in CTR of other articles in the same subcategory
- entity_recent_ctr_mean_6h: Average recent CTR of other articles sharing the candidate's entities
- entity_ctr_momentum_mean_6h: Average CTR momentum across the candidate's entities
- entity_ctr_momentum_max_6h: Maximum CTR momentum among the candidate's entities
- semantic_neighbor_ctr: CTR of semantically similar articles, weighted by similarity and calculated using only behaviour available before prediction time
- content_prior_ctr: Combined CTR estimate from category, subcategory, entity, and semantic-neighbour behaviour
- content_smoothed_ctr: Blends the article's own historical behaviour with content_prior_ctr, relying more on borrowed behaviour when the article has few exposures and more on its own behaviour when it becomes warm

Tier 3 Feature Names:
- category: Main news category
- subcategory: More specific news subcategory
- entity_count: Number of linked entities in the article
- entity_embedding_coverage: Proportion of linked entities with available pretrained entity embeddings
- title_embedding: Sentence Transformer embedding of the article title (Intermediate Feature, NOT USED for Training)
- abstract_embedding: Sentence Transformer embedding of the article abstract (Intermediate Feature, NOT USED for Training)
- article_text_embedding: Combined title and abstract text representation (Intermediate Feature, NOT USED for Training)
- article_entity_embedding: Average embedding of the article's linked entities (Intermediate Feature, NOT USED for Training)

Tier U Feature Names:
- history_length: Number of articles in the user's clicked history
- category_affinity: Fraction of the user's clicked history belonging to the candidate's category
- subcategory_affinity: Fraction of the user's clicked history belonging to the candidate's subcategory
- user_text_embedding: Average text embedding of articles in the user's clicked history (Intermediate Feature, NOT USED for Training)
- user_entity_embedding: Average entity embedding of articles in the user's clicked history (Intermediate Feature, NOT USED for Training)
- text_similarity: Cosine similarity between the user's text profile and the candidate's text embedding
- entity_similarity: Cosine similarity between the user's entity profile and the candidate's entity embedding
- entity_overlap_ratio: Proportion of the candidate's exact entities that have appeared in the user's clicked history

Evaluation Features Name: These are not used as model inputs. They are used to measure performance across cold and warm articles.
- cold_flag: Whether the article has fewer than 5 prior exposures
- zero_prior_click_flag: Whether the article has received zero prior clicks
- exposure_bucket: Article exposure group: 0, 1–4, 5–20, 21–100, or >100

In [1]:
from pathlib import Path

cache_path = (
    Path.home()
    / ".cache"
    / "kagglehub"
    / "datasets"
    / "arashnic"
    / "mind-news-dataset"
)

print(cache_path)
print(cache_path.exists())

C:\Users\andre\.cache\kagglehub\datasets\arashnic\mind-news-dataset
True


In [ ]:
train_folders = list(cache_path.rglob("MINDsmall_train"))

print(train_folders)

[WindowsPath('C:/Users/andre/.cache/kagglehub/datasets/arashnic/mind-news-dataset/versions/2/MINDsmall_train')]


In [3]:
train_path = train_folders[0]

print(list(train_path.iterdir()))

[WindowsPath('C:/Users/andre/.cache/kagglehub/datasets/arashnic/mind-news-dataset/versions/2/MINDsmall_train/behaviors.tsv'), WindowsPath('C:/Users/andre/.cache/kagglehub/datasets/arashnic/mind-news-dataset/versions/2/MINDsmall_train/entity_embedding.vec'), WindowsPath('C:/Users/andre/.cache/kagglehub/datasets/arashnic/mind-news-dataset/versions/2/MINDsmall_train/news.tsv'), WindowsPath('C:/Users/andre/.cache/kagglehub/datasets/arashnic/mind-news-dataset/versions/2/MINDsmall_train/relation_embedding.vec')]


In [4]:
import pandas as pd

behaviors = pd.read_csv(
    train_path / "behaviors.tsv",
    sep="\t",
    header=None,
    names=[
        "impression_id",
        "user_id",
        "time",
        "history",
        "impressions"
    ]
)

news = pd.read_csv(
    train_path / "news.tsv",
    sep="\t",
    header=None,
    names=[
        "news_id",
        "category",
        "subcategory",
        "title",
        "abstract",
        "url",
        "title_entities",
        "abstract_entities"
    ]
)

print(behaviors.shape)
print(news.shape)

(156965, 5)
(51282, 8)


In [5]:
behaviors.head()

,impression_id,user_id,time,history,impressions
0,1,U13740,11/11/2019 9:05:58 AM,N55189 N42782 N34694 N45794 N18445 N63302 N104...,N55689-1 N35729-0
1,2,U91836,11/12/2019 6:11:30 PM,N31739 N6072 N63045 N23979 N35656 N43353 N8129...,N20678-0 N39317-0 N58114-0 N20495-0 N42977-0 N...
2,3,U73700,11/14/2019 7:01:48 AM,N10732 N25792 N7563 N21087 N41087 N5445 N60384...,N50014-0 N23877-0 N35389-0 N49712-0 N16844-0 N...
3,4,U34670,11/11/2019 5:28:05 AM,N45729 N2203 N871 N53880 N41375 N43142 N33013 ...,N35729-0 N33632-0 N49685-1 N27581-0
4,5,U8125,11/12/2019 4:11:21 PM,N10078 N56514 N14904 N33740,N39985-0 N36050-0 N16096-0 N8400-1 N22407-0 N6...


In [6]:
news.head()

,news_id,category,subcategory,title,abstract,url,title_entities,abstract_entities
0,N55528,lifestyle,lifestyleroyals,"The Brands Queen Elizabeth, Prince Charles, an...","Shop the notebooks, jackets, and more that the...",https://assets.msn.com/labs/mind/AAGH0ET.html,"[{""Label"": ""Prince Philip, Duke of Edinburgh"",...",[]
1,N19639,health,weightloss,50 Worst Habits For Belly Fat,These seemingly harmless habits are holding yo...,https://assets.msn.com/labs/mind/AAB19MK.html,"[{""Label"": ""Adipose tissue"", ""Type"": ""C"", ""Wik...","[{""Label"": ""Adipose tissue"", ""Type"": ""C"", ""Wik..."
2,N61837,news,newsworld,The Cost of Trump's Aid Freeze in the Trenches...,Lt. Ivan Molchanets peeked over a parapet of s...,https://assets.msn.com/labs/mind/AAJgNsz.html,[],"[{""Label"": ""Ukraine"", ""Type"": ""G"", ""WikidataId..."
3,N53526,health,voices,I Was An NBA Wife. Here's How It Affected My M...,"I felt like I was a fraud, and being an NBA wi...",https://assets.msn.com/labs/mind/AACk2N6.html,[],"[{""Label"": ""National Basketball Association"", ..."
4,N38324,health,medical,"How to Get Rid of Skin Tags, According to a De...","They seem harmless, but there's a very good re...",https://assets.msn.com/labs/mind/AAAKEkt.html,"[{""Label"": ""Skin tag"", ""Type"": ""C"", ""WikidataI...","[{""Label"": ""Skin tag"", ""Type"": ""C"", ""WikidataI..."


Explode Dataset:

Step 1: Create one row for every user × candidate article × impression time
Step 2: Attach category and subcategory

Things to note:
- "label" column indicates whether article was clicked by user or not
- 156,965 original impression rows expanded into 5,843,444 individual user–candidate examples
- 5,607,100 rows not clicked, 236,344 rows clicked

In [7]:
import pandas as pd
import numpy as np
import json
import os

beh = behaviors.copy().reset_index(drop=True)
news_df = news.copy().drop_duplicates("news_id").reset_index(drop=True)

beh["time"] = pd.to_datetime(beh["time"])

#Unique ID for each original impression row
beh["behavior_row_id"] = np.arange(len(beh))

#Split impression string into individual candidate articles
base = beh[
    [
        "behavior_row_id",
        "impression_id",
        "user_id",
        "time",
        "impressions"
    ]
].copy()

base["candidate"] = base["impressions"].str.split()

candidate_df = (
    base
    .explode("candidate", ignore_index=True)
    .drop(columns="impressions")
)

#MIND format: N12345-1 or N12345-0
parsed = candidate_df["candidate"].str.rsplit("-", n=1, expand=True)

candidate_df["news_id"] = parsed[0]

#Label just means whether got clicked or not
candidate_df["label"] = parsed[1].astype("int8")

candidate_df = candidate_df.drop(columns="candidate")

candidate_df["candidate_row_id"] = np.arange(len(candidate_df))

candidate_df.head(10)

,behavior_row_id,impression_id,user_id,time,news_id,label,candidate_row_id
0,0,1,U13740,2019-11-11 09:05:58,N55689,1,0
1,0,1,U13740,2019-11-11 09:05:58,N35729,0,1
2,1,2,U91836,2019-11-12 18:11:30,N20678,0,2
3,1,2,U91836,2019-11-12 18:11:30,N39317,0,3
4,1,2,U91836,2019-11-12 18:11:30,N58114,0,4
5,1,2,U91836,2019-11-12 18:11:30,N20495,0,5
6,1,2,U91836,2019-11-12 18:11:30,N42977,0,6
7,1,2,U91836,2019-11-12 18:11:30,N22407,0,7
8,1,2,U91836,2019-11-12 18:11:30,N14592,0,8
9,1,2,U91836,2019-11-12 18:11:30,N17059,1,9


In [8]:
category_lookup = news_df.set_index("news_id")["category"]
subcategory_lookup = news_df.set_index("news_id")["subcategory"]

candidate_df["category"] = candidate_df["news_id"].map(category_lookup)
candidate_df["subcategory"] = candidate_df["news_id"].map(subcategory_lookup)

print(candidate_df.head())
print("Candidate rows:", len(candidate_df))
print("Labels:", candidate_df["label"].value_counts())

#Check if MINDs labels were parsed correctly
assert candidate_df["label"].isin([0, 1]).all()

#Check if every row received its own unique ID
assert candidate_df["candidate_row_id"].is_unique

   behavior_row_id  impression_id user_id                time news_id  label  \
0                0              1  U13740 2019-11-11 09:05:58  N55689      1   
1                0              1  U13740 2019-11-11 09:05:58  N35729      0   
2                1              2  U91836 2019-11-12 18:11:30  N20678      0   
3                1              2  U91836 2019-11-12 18:11:30  N39317      0   
4                1              2  U91836 2019-11-12 18:11:30  N58114      0   

   candidate_row_id category   subcategory  
0                 0   sports  football_nfl  
1                 1     news        newsus  
2                 2   sports   more_sports  
3                 3     news  newspolitics  
4                 4    autos     autosnews  
Candidate rows: 5843444
Labels: label
0    5607100
1     236344
Name: count, dtype: int64


Tier 1: Article-level behavioural features

In [9]:
article_time = (
    candidate_df
    .groupby(["news_id", "time"], as_index=False)
    .agg(
        exposures_at_time=("label", "size"),
        clicks_at_time=("label", "sum")
    ).sort_values(["news_id", "time"]))

In [10]:
article_time["prior_exposures"] = (
    article_time
    .groupby("news_id")["exposures_at_time"]
    .cumsum()
    - article_time["exposures_at_time"])

article_time["prior_clicks"] = (
    article_time
    .groupby("news_id")["clicks_at_time"]
    .cumsum()
    - article_time["clicks_at_time"])

article_time["prior_ctr"] = np.where(
    article_time["prior_exposures"] > 0,
    article_time["prior_clicks"] / article_time["prior_exposures"],
    np.nan)

In [11]:
article_time["first_seen_time"] = (
    article_time
    .groupby("news_id")["time"]
    .transform("min"))

article_time["article_age_hours"] = ((article_time["time"] - article_time["first_seen_time"]).dt.total_seconds() / 3600)

Recent 6 hours and 12 hours (can be changed) exposures and clicks

In [12]:
def add_rolling_windows(df, group_col):
    def process_group(g):
        #Save the group value, e.g. the news_id
        group_value = g.name

        g = g.sort_values("time").copy()
        g = g.set_index("time")

        #Most recent 6 hours, excluding current timestamp
        g["recent_exposures_6h"] = (
            g["exposures_at_time"]
            .rolling("6h", closed="left")
            .sum()
            .fillna(0)
        )

        g["recent_clicks_6h"] = (
            g["clicks_at_time"]
            .rolling("6h", closed="left")
            .sum()
            .fillna(0)
        )

        #Previous 12 hours
        exposures_12h = (
            g["exposures_at_time"]
            .rolling("12h", closed="left")
            .sum()
            .fillna(0)
        )

        clicks_12h = (
            g["clicks_at_time"]
            .rolling("12h", closed="left")
            .sum()
            .fillna(0)
        )

        # Previous 6 hours = 12h window - latest 6h
        g["previous_exposures_6h"] = (
            exposures_12h
            - g["recent_exposures_6h"]
        )

        g["previous_clicks_6h"] = (
            clicks_12h
            - g["recent_clicks_6h"]
        )

        g = g.reset_index()

        #IMPORTANT: restore news_id/category/etc.
        g[group_col] = group_value

        return g

    return (
        df
        .groupby(group_col, group_keys=False)
        .apply(
            process_group,
            include_groups=False
        )
        .reset_index(drop=True)
    )

In [13]:
article_time = add_rolling_windows(
    article_time,
    "news_id"
)

In [14]:
article_time["recent_ctr_6h"] = np.where(
    article_time["recent_exposures_6h"] > 0,
    article_time["recent_clicks_6h"]
    / article_time["recent_exposures_6h"],
    np.nan
)

article_time["previous_ctr_6h"] = np.where(
    article_time["previous_exposures_6h"] > 0,
    article_time["previous_clicks_6h"]
    / article_time["previous_exposures_6h"],
    np.nan
)

article_time["ctr_momentum_6h"] = (article_time["recent_ctr_6h"] - article_time["previous_ctr_6h"])

In [15]:
candidate_df = candidate_df.merge(
    article_time[
        [
            "news_id",
            "time",
            "first_seen_time",
            "article_age_hours",
            "prior_exposures",
            "prior_clicks",
            "prior_ctr",
            "recent_exposures_6h",
            "recent_clicks_6h",
            "recent_ctr_6h",
            "previous_exposures_6h",
            "previous_clicks_6h",
            "previous_ctr_6h",
            "ctr_momentum_6h"
        ]
    ],
    on=["news_id", "time"],
    how="left"
)

Behaviour Confidence

Using k=20 for now

In [16]:
K = 20

candidate_df["behavior_confidence"] = (candidate_df["prior_exposures"] / (candidate_df["prior_exposures"] + K))

In [17]:
assert (candidate_df["article_age_hours"] >= 0).all()
assert (candidate_df["prior_exposures"] >= 0).all()
assert (candidate_df["prior_clicks"] >= 0).all()

assert (
    candidate_df["prior_clicks"]
    <= candidate_df["prior_exposures"]
).all()

assert candidate_df["behavior_confidence"].between(0, 1).all()

assert candidate_df["prior_ctr"].dropna().between(0, 1).all()
assert candidate_df["recent_ctr_6h"].dropna().between(0, 1).all()

In [18]:
candidate_df[
    [
        "news_id",
        "time",
        "article_age_hours",
        "prior_exposures",
        "prior_clicks",
        "prior_ctr",
        "behavior_confidence",
        "recent_exposures_6h",
        "recent_clicks_6h",
        "recent_ctr_6h",
        "ctr_momentum_6h"
    ]
].head(30)

,news_id,time,article_age_hours,prior_exposures,prior_clicks,prior_ctr,behavior_confidence,recent_exposures_6h,recent_clicks_6h,recent_ctr_6h,ctr_momentum_6h
0,N55689,2019-11-11 09:05:58,14.153611,5657,2369,0.418773,0.996477,5656.0,2369.0,0.418847,NaN
1,N35729,2019-11-11 09:05:58,27.303056,11109,2578,0.232064,0.998203,9378.0,2263.0,0.241309,0.010360
2,N20678,2019-11-12 18:11:30,10.112500,4923,462,0.093845,0.995954,4905.0,461.0,0.093986,0.038430
3,N39317,2019-11-12 18:11:30,12.999444,3913,345,0.088168,0.994915,3505.0,308.0,0.087874,-0.003710
4,N58114,2019-11-12 18:11:30,15.959444,1936,30,0.015496,0.989775,1909.0,30.0,0.015715,0.015715
5,N20495,2019-11-12 18:11:30,2.056944,1847,287,0.155387,0.989288,1847.0,287.0,0.155387,NaN
6,N42977,2019-11-12 18:11:30,2.181944,1592,210,0.131910,0.987593,1592.0,210.0,0.131910,NaN
7,N22407,2019-11-12 18:11:30,12.842500,1949,35,0.017958,0.989843,1451.0,26.0,0.017919,-0.000337
8,N14592,2019-11-12 18:11:30,11.750833,1718,26,0.015134,0.988493,1711.0,25.0,0.014611,-0.128246
9,N17059,2019-11-12 18:11:30,5.400833,3403,219,0.064355,0.994157,3403.0,219.0,0.064355,NaN


In [19]:
row = candidate_df.sample(1, random_state=42).iloc[0]

manual = candidate_df[
    (candidate_df["news_id"] == row["news_id"]) &
    (candidate_df["time"] < row["time"])
]

print("Article:", row["news_id"])
print("Prediction time:", row["time"])

print("Calculated prior exposures:", row["prior_exposures"])
print("Manual prior exposures:", len(manual))

print("Calculated prior clicks:", row["prior_clicks"])
print("Manual prior clicks:", manual["label"].sum())

Article: N23805
Prediction time: 2019-11-14 07:33:18
Calculated prior exposures: 962
Manual prior exposures: 962
Calculated prior clicks: 60
Manual prior clicks: 60


Category Transferred Behaviour

Group candidate rows by category then time

So something like this:
time       news_id   category   label
10:00      N1        sports       1
10:00      N2        sports       0
10:00      N3        news         0
10:05      N4        sports       1

Becomes:
category   time    exposures_at_time   clicks_at_time
sports     10:00          2                  1
news       10:00          1                  0
sports     10:05          1                  1

becuase number of rows = number of exposures and doing sum(label) = number of clicks

In [20]:
category_time = (
    candidate_df
    .groupby(["category", "time"], as_index=False)
    .agg(
        exposures_at_time=("label", "size"),
        clicks_at_time=("label", "sum")
    )
)

In [21]:
category_time = add_rolling_windows(
    category_time,
    "category"
)

Rename to clarify whether clicks and exposures refer to whole category or the individual candidate article

In [22]:
category_time = category_time.rename(columns={
    "recent_exposures_6h": "category_recent_exposures_all",
    "recent_clicks_6h": "category_recent_clicks_all",
    "previous_exposures_6h": "category_previous_exposures_all",
    "previous_clicks_6h": "category_previous_clicks_all"
})

In [23]:
candidate_df = candidate_df.merge(
    category_time[
        [
            "category",
            "time",
            "category_recent_exposures_all",
            "category_recent_clicks_all",
            "category_previous_exposures_all",
            "category_previous_clicks_all"
        ]
    ],
    on=["category", "time"],
    how="left"
)

Remove candidate article's own clicks or exposures when calculating exposure or clicks of its category

In [24]:
candidate_df["category_recent_exposures_other"] = (
    candidate_df["category_recent_exposures_all"]
    - candidate_df["recent_exposures_6h"]
).clip(lower=0)

candidate_df["category_recent_clicks_other"] = (
    candidate_df["category_recent_clicks_all"]
    - candidate_df["recent_clicks_6h"]
).clip(lower=0)

In [25]:
candidate_df["category_previous_exposures_other"] = (
    candidate_df["category_previous_exposures_all"]
    - candidate_df["previous_exposures_6h"]
).clip(lower=0)

candidate_df["category_previous_clicks_other"] = (
    candidate_df["category_previous_clicks_all"]
    - candidate_df["previous_clicks_6h"]
).clip(lower=0)

In [26]:
#Calculate ctr
candidate_df["category_recent_ctr_6h"] = np.where(
    candidate_df["category_recent_exposures_other"] > 0,
    candidate_df["category_recent_clicks_other"]
    / candidate_df["category_recent_exposures_other"],
    np.nan
)

candidate_df["category_previous_ctr_6h"] = np.where(
    candidate_df["category_previous_exposures_other"] > 0,
    candidate_df["category_previous_clicks_other"]
    / candidate_df["category_previous_exposures_other"],
    np.nan
)

candidate_df["category_ctr_momentum_6h"] = (
    candidate_df["category_recent_ctr_6h"]
    - candidate_df["category_previous_ctr_6h"]
)

In [27]:
#Verifying if everything worked correctly
assert candidate_df[
    "category_recent_ctr_6h"
].dropna().between(0, 1).all()

candidate_df[
    [
        "news_id",
        "category",
        "category_recent_ctr_6h",
        "category_ctr_momentum_6h"
    ]
].head(30)

,news_id,category,category_recent_ctr_6h,category_ctr_momentum_6h
0,N55689,sports,0.037806,-0.015127
1,N35729,news,0.028111,-0.006157
2,N20678,sports,0.055945,0.007416
3,N39317,news,0.034031,-0.018586
4,N58114,autos,0.028566,0.001382
5,N20495,news,0.033681,-0.019052
6,N42977,news,0.034356,-0.018376
7,N22407,finance,0.054346,-0.033122
8,N14592,entertainment,0.033915,-0.013436
9,N17059,finance,0.052586,-0.034143


Subcategory Transferred Behaviour

In [28]:
subcategory_time = (
    candidate_df
    .groupby(["subcategory", "time"], as_index=False)
    .agg(
        exposures_at_time=("label", "size"),
        clicks_at_time=("label", "sum")
    )
)

subcategory_time = add_rolling_windows(
    subcategory_time,
    "subcategory"
)

In [29]:
subcategory_time = subcategory_time.rename(columns={
    "recent_exposures_6h": "subcategory_recent_exposures_all",
    "recent_clicks_6h": "subcategory_recent_clicks_all",
    "previous_exposures_6h": "subcategory_previous_exposures_all",
    "previous_clicks_6h": "subcategory_previous_clicks_all"
})

In [30]:
candidate_df = candidate_df.merge(
    subcategory_time[
        [
            "subcategory",
            "time",
            "subcategory_recent_exposures_all",
            "subcategory_recent_clicks_all",
            "subcategory_previous_exposures_all",
            "subcategory_previous_clicks_all"
        ]
    ],
    on=["subcategory", "time"],
    how="left"
)

In [31]:
sub_recent_exp = (
    candidate_df["subcategory_recent_exposures_all"]
    - candidate_df["recent_exposures_6h"]
).clip(lower=0)

sub_recent_click = (
    candidate_df["subcategory_recent_clicks_all"]
    - candidate_df["recent_clicks_6h"]
).clip(lower=0)

sub_previous_exp = (
    candidate_df["subcategory_previous_exposures_all"]
    - candidate_df["previous_exposures_6h"]
).clip(lower=0)

sub_previous_click = (
    candidate_df["subcategory_previous_clicks_all"]
    - candidate_df["previous_clicks_6h"]
).clip(lower=0)

In [32]:
candidate_df["subcategory_recent_ctr_6h"] = np.where(
    sub_recent_exp > 0,
    sub_recent_click / sub_recent_exp,
    np.nan
)

subcategory_previous_ctr = np.where(
    sub_previous_exp > 0,
    sub_previous_click / sub_previous_exp,
    np.nan
)

candidate_df["subcategory_ctr_momentum_6h"] = (
    candidate_df["subcategory_recent_ctr_6h"]
    - subcategory_previous_ctr
)

In [33]:
assert candidate_df[
    "subcategory_recent_ctr_6h"
].dropna().between(0, 1).all()

Entity Transferred Behaviour

In [34]:
def extract_entity_ids(value):
    if pd.isna(value):
        return []

    try:
        entities = json.loads(value)

        return [
            entity["WikidataId"]
            for entity in entities
            if entity.get("WikidataId")
        ]

    except:
        return []

In [35]:
news_df["title_entity_ids"] = (
    news_df["title_entities"]
    .apply(extract_entity_ids)
)

news_df["abstract_entity_ids"] = (
    news_df["abstract_entities"]
    .apply(extract_entity_ids)
)

In [36]:
news_df["entity_ids"] = news_df.apply(
    lambda row: list(
        set(
            row["title_entity_ids"]
            + row["abstract_entity_ids"]
        )
    ),
    axis=1
)

In [37]:
entity_list_lookup = dict(
    zip(
        news_df["news_id"],
        news_df["entity_ids"]
    )
)

In [38]:
entity_events = candidate_df[
    [
        "candidate_row_id",
        "news_id",
        "time",
        "label"
    ]
].copy()

entity_events["entity_id"] = (
    entity_events["news_id"]
    .map(entity_list_lookup)
)

entity_events = (
    entity_events
    .explode("entity_id")
    .dropna(subset=["entity_id"])
)

In [39]:
entity_time = (
    entity_events
    .groupby(["entity_id", "time"], as_index=False)
    .agg(
        exposures_at_time=("label", "size"),
        clicks_at_time=("label", "sum")
    )
)

entity_time = add_rolling_windows(
    entity_time,
    "entity_id"
)

In [40]:
entity_time = entity_time.rename(columns={
    "recent_exposures_6h": "entity_recent_exposures_all",
    "recent_clicks_6h": "entity_recent_clicks_all",
    "previous_exposures_6h": "entity_previous_exposures_all",
    "previous_clicks_6h": "entity_previous_clicks_all"
})

In [41]:
candidate_entities = candidate_df[
    [
        "candidate_row_id",
        "news_id",
        "time",
        "recent_exposures_6h",
        "recent_clicks_6h",
        "previous_exposures_6h",
        "previous_clicks_6h"
    ]
].copy()

candidate_entities["entity_id"] = (
    candidate_entities["news_id"]
    .map(entity_list_lookup)
)

candidate_entities = (
    candidate_entities
    .explode("entity_id")
    .dropna(subset=["entity_id"])
)

In [42]:
candidate_entities = candidate_entities.merge(
    entity_time[
        [
            "entity_id",
            "time",
            "entity_recent_exposures_all",
            "entity_recent_clicks_all",
            "entity_previous_exposures_all",
            "entity_previous_clicks_all"
        ]
    ],
    on=["entity_id", "time"],
    how="left"
)

In [43]:
candidate_entities["entity_recent_exp_other"] = (
    candidate_entities["entity_recent_exposures_all"]
    - candidate_entities["recent_exposures_6h"]
).clip(lower=0)

candidate_entities["entity_recent_click_other"] = (
    candidate_entities["entity_recent_clicks_all"]
    - candidate_entities["recent_clicks_6h"]
).clip(lower=0)

candidate_entities["entity_previous_exp_other"] = (
    candidate_entities["entity_previous_exposures_all"]
    - candidate_entities["previous_exposures_6h"]
).clip(lower=0)

candidate_entities["entity_previous_click_other"] = (
    candidate_entities["entity_previous_clicks_all"]
    - candidate_entities["previous_clicks_6h"]
).clip(lower=0)

In [44]:
candidate_entities["entity_recent_ctr"] = np.where(
    candidate_entities["entity_recent_exp_other"] > 0,
    candidate_entities["entity_recent_click_other"]
    / candidate_entities["entity_recent_exp_other"],
    np.nan
)

candidate_entities["entity_previous_ctr"] = np.where(
    candidate_entities["entity_previous_exp_other"] > 0,
    candidate_entities["entity_previous_click_other"]
    / candidate_entities["entity_previous_exp_other"],
    np.nan
)

candidate_entities["entity_ctr_momentum"] = (
    candidate_entities["entity_recent_ctr"]
    - candidate_entities["entity_previous_ctr"]
)

In [45]:
entity_features = (
    candidate_entities
    .groupby("candidate_row_id")
    .agg(
        entity_recent_ctr_mean_6h=(
            "entity_recent_ctr",
            "mean"
        ),
        entity_ctr_momentum_mean_6h=(
            "entity_ctr_momentum",
            "mean"
        ),
        entity_ctr_momentum_max_6h=(
            "entity_ctr_momentum",
            "max"
        )
    )
    .reset_index()
)

In [46]:
candidate_df = candidate_df.merge(
    entity_features,
    on="candidate_row_id",
    how="left"
)

In [47]:
assert candidate_df[
    "entity_recent_ctr_mean_6h"
].dropna().between(0, 1).all()

In [48]:
candidate_df[
    [
        "news_id",
        "entity_recent_ctr_mean_6h",
        "entity_ctr_momentum_mean_6h",
        "entity_ctr_momentum_max_6h"
    ]
].dropna().head(30)

,news_id,entity_recent_ctr_mean_6h,entity_ctr_momentum_mean_6h,entity_ctr_momentum_max_6h
0,N55689,0.054398,0.042129,0.042129
1,N35729,0.057820,-0.002364,-0.001654
3,N39317,0.021874,-0.008738,-0.008643
5,N20495,0.027489,-0.022172,0.004023
8,N14592,0.021073,0.021073,0.021073
9,N17059,0.098118,-0.126387,-0.126387
10,N33677,0.072243,0.050815,0.050815
11,N7821,0.110730,0.030561,0.033622
12,N6890,0.098361,0.098361,0.098361
13,N50014,0.023799,0.013518,0.013518


Pure Content Features

In [49]:
#Count number of entities
news_df["entity_count"] = (
    news_df["entity_ids"]
    .apply(len)
)

#Load MINDS provided entity embeddings
entity_embedding_path = os.path.join(
    train_path,
    "entity_embedding.vec"
)

entity_vectors = {}

with open(
    entity_embedding_path,
    "r",
    encoding="utf-8"
) as f:

    for line in f:
        parts = line.strip().split("\t")

        entity_id = parts[0]

        vector = np.array(
            parts[1:],
            dtype=np.float32
        )

        entity_vectors[entity_id] = vector

first_vector = next(iter(entity_vectors.values()))

print("Number of entity vectors:", len(entity_vectors))
print("Vector dimension:", len(first_vector))

assert len(first_vector) == 100

Number of entity vectors: 26904
Vector dimension: 100


Entity coverage, checking how many entites of each entity in news.tsv have an existing embedding

In [50]:
def entity_coverage(entity_ids):
    if len(entity_ids) == 0:
        return np.nan

    available = sum(
        entity_id in entity_vectors
        for entity_id in entity_ids
    )

    return available / len(entity_ids)

news_df["entity_embedding_coverage"] = (
    news_df["entity_ids"]
    .apply(entity_coverage)
)

assert news_df[
    "entity_embedding_coverage"
].dropna().between(0, 1).all()

news_df[
    [
        "news_id",
        "entity_count",
        "entity_embedding_coverage"
    ]
].head(20)

,news_id,entity_count,entity_embedding_coverage
0,N55528,3,1.0
1,N19639,1,1.0
2,N61837,1,1.0
3,N53526,1,1.0
4,N38324,3,1.0
5,N2073,1,1.0
6,N49186,1,1.0
7,N59295,2,1.0
8,N24510,1,1.0
9,N39237,2,1.0


For each article, take all the available entity embeddings provided from MINDs and average them, so each article gets a single 100-dimension entity representation

In [51]:
def build_article_entity_embedding(entity_ids):
    vectors = [
        entity_vectors[e]
        for e in entity_ids
        if e in entity_vectors
    ]

    if len(vectors) == 0:
        return np.zeros(100, dtype=np.float32)

    return np.mean(vectors, axis=0)

article_entity_embeddings = np.vstack(
    news_df["entity_ids"]
    .apply(build_article_entity_embedding)
)

print(article_entity_embeddings.shape)

#Verify
assert article_entity_embeddings.shape == (
    len(news_df),
    100
)

(51282, 100)


In [52]:
#%pip install sentence-transformers

Title and abstract embeddings. MINDs only gives them as raw data, so create embeddings use Sentence Transformers, that converts text into numerical vectors.
Main idea is create title and abstract embeddings individually as they may contain different levels of detail. Then average them such that earch article only carries one overall text representation.
Then normalise them each to have length 1 for later use of cosine similarity.

For Sentence Transformer, use all-MiniLM-L6-v2 because it is designed to turn sentences into numerical vectors that can be compared for semantic similarity. This fits our use case because we need to compare article titles and abstracts, find similar articles, and measure how well a candidate article matches a user’s reading history. We chose MiniLM because it gives compact 384-dimensional embeddings and is much faster than larger alternatives, while still providing good semantic representations.

Other models available and why we did not choose them:
all-mpnet-base-v2: better embedding quality with 768 embedding size, but slower and more computationally expensive - do not think such level of complexity is needed for this

TF-IDF: mainly capures shared words, which is not the purpose here

Bert: produces token-level contextual representations. You then need to decide how to pool those tokens into one article vector, and vanilla BERT was not specifically trained so that cosine distance between those sentence vectors directly represents semantic similarity

In [53]:
from sentence_transformers import SentenceTransformer

text_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

title_embeddings = text_model.encode(
    news_df["title"].fillna("").tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

abstract_embeddings = text_model.encode(
    news_df["abstract"].fillna("").tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

article_text_embeddings = (
    title_embeddings
    + abstract_embeddings
) / 2

norms = np.linalg.norm(
    article_text_embeddings,
    axis=1,
    keepdims=True
)

norms[norms == 0] = 1

article_text_embeddings = (
    article_text_embeddings / norms
)

print("Title:", title_embeddings.shape)
print("Abstract:", abstract_embeddings.shape)
print("Combined:", article_text_embeddings.shape)

c:\Users\andre\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Batches: 100%|██████████| 802/802 [05:28<00:00,  2.44it/s]


Title: (51282, 384)
Abstract: (51282, 384)
Combined: (51282, 384)


In [54]:
#Create article lookups
#Basically just a dictionary to find text or entity vector by article title
news_index = {
    news_id: i
    for i, news_id in enumerate(news_df["news_id"])
}

article_entity_lookup = {
    news_id: article_entity_embeddings[i]
    for i, news_id in enumerate(news_df["news_id"])
}

article_text_lookup = {
    news_id: article_text_embeddings[i]
    for i, news_id in enumerate(news_df["news_id"])
}

User Representation, using behaviours.history

In [55]:
#History length
#First part just turns a history which is given as a string like "N20, N1, N53" into ["N20", "N1", "N53"] to count number
beh["history_ids"] = (
    beh["history"]
    .fillna("")
    .apply(
        lambda x: x.split() if x.strip() else []
    )
)

beh["history_length"] = (
    beh["history_ids"]
    .apply(len)
)

#Merge with candidate df
candidate_df = candidate_df.merge(
    beh[
        [
            "behavior_row_id",
            "history_length"
        ]
    ],
    on="behavior_row_id",
    how="left"
)

User text embedding
For each article in user's history, get their 384-dimension text embedding and take the average of them.
Eg, if a user's history is mostly football articles, resulting vector should roughly point towards football related smemantic content.
Then normalise it so its length its 1 for cosine similarity calculation.

In [56]:
TEXT_DIM = article_text_embeddings.shape[1]

user_text_embeddings = np.zeros(
    (len(beh), TEXT_DIM),
    dtype=np.float32
)

for i, history in enumerate(beh["history_ids"]):

    vectors = [
        article_text_lookup[n]
        for n in history
        if n in article_text_lookup
    ]

    if vectors:
        user_text_embeddings[i] = np.mean(
            vectors,
            axis=0
        )

#Normalise
norms = np.linalg.norm(
    user_text_embeddings,
    axis=1,
    keepdims=True
)

norms[norms == 0] = 1

user_text_embeddings /= norms

#Verify
print(user_text_embeddings.shape)

assert user_text_embeddings.shape[0] == len(beh)

(156965, 384)


User entity embedding
Same as above but using 100-dimension entity embeddings instead.
Every user should have a user_text_embedding and user_entity_embedding.

In [57]:
user_entity_embeddings = np.zeros(
    (len(beh), 100),
    dtype=np.float32
)

for i, history in enumerate(beh["history_ids"]):

    vectors = [
        article_entity_lookup[n]
        for n in history
        if n in article_entity_lookup
        and np.linalg.norm(article_entity_lookup[n]) > 0
    ]

    if vectors:
        user_entity_embeddings[i] = np.mean(
            vectors,
            axis=0
        )

#Normalise
norms = np.linalg.norm(
    user_entity_embeddings,
    axis=1,
    keepdims=True
)

norms[norms == 0] = 1

user_entity_embeddings /= norms

Category Affinity

Basically just means among the user's entire history, what's the proportion of each category, eg user history has 5 articles and 3 of which are sports, so this user's sports category affinity is 0.6. Basically measures how often has this user historically clicked this type of article

How it is done is code first explodes the user's history into one row per clicked article, maps each article to its category, then counts the fraction belonging to the candidate's category.

In [58]:
history_long = (
    beh[
        ["behavior_row_id", "history_ids"]
    ]
    .explode("history_ids")
    .rename(columns={
        "history_ids": "history_news_id"
    })
)

history_long = history_long[
    history_long["history_news_id"].notna()
    & (history_long["history_news_id"] != "")
]

#Map categories
history_long["category"] = (
    history_long["history_news_id"]
    .map(category_lookup)
)

history_long["subcategory"] = (
    history_long["history_news_id"]
    .map(subcategory_lookup)
)

category_affinity = (
    history_long
    .groupby(
        ["behavior_row_id", "category"]
    )
    .size()
    .reset_index(name="category_click_count")
)

#Merge with candidate df
category_affinity = category_affinity.merge(
    beh[
        ["behavior_row_id", "history_length"]
    ],
    on="behavior_row_id"
)

category_affinity["category_affinity"] = (
    category_affinity["category_click_count"]
    / category_affinity["history_length"]
)

candidate_df = candidate_df.merge(
    category_affinity[
        [
            "behavior_row_id",
            "category",
            "category_affinity"
        ]
    ],
    on=["behavior_row_id", "category"],
    how="left"
)

candidate_df["category_affinity"] = (
    candidate_df["category_affinity"]
    .fillna(0)
)

Subcategory affinity

Same logic as category affinity but for subcategory

In [59]:
subcategory_affinity = (
    history_long
    .groupby(
        ["behavior_row_id", "subcategory"]
    )
    .size()
    .reset_index(name="subcategory_click_count")
)

subcategory_affinity = subcategory_affinity.merge(
    beh[
        ["behavior_row_id", "history_length"]
    ],
    on="behavior_row_id"
)

subcategory_affinity["subcategory_affinity"] = (
    subcategory_affinity["subcategory_click_count"]
    / subcategory_affinity["history_length"]
)

#Merge with candidate df
candidate_df = candidate_df.merge(
    subcategory_affinity[
        [
            "behavior_row_id",
            "subcategory",
            "subcategory_affinity"
        ]
    ],
    on=["behavior_row_id", "subcategory"],
    how="left"
)

candidate_df["subcategory_affinity"] = (
    candidate_df["subcategory_affinity"]
    .fillna(0)
)

#Verify both category and subcategory affinity
assert candidate_df[
    "category_affinity"
].between(0, 1).all()

assert candidate_df[
    "subcategory_affinity"
].between(0, 1).all()

User-Article text and entity similarity

In [60]:
#candidate_df is large, calculate these in chunks rather than storing vectors inside every row.
def add_similarity_features(df, chunk_size=100000):

    text_scores = np.zeros(
        len(df),
        dtype=np.float32
    )

    entity_scores = np.zeros(
        len(df),
        dtype=np.float32
    )

    news_indices = (
        df["news_id"]
        .map(news_index)
        .to_numpy()
    )

    user_indices = (
        df["behavior_row_id"]
        .to_numpy()
    )

    for start in range(0, len(df), chunk_size):

        end = min(
            start + chunk_size,
            len(df)
        )

        n_idx = news_indices[start:end]
        u_idx = user_indices[start:end]

        # Text cosine similarity
        text_scores[start:end] = np.sum(
            user_text_embeddings[u_idx]
            * article_text_embeddings[n_idx],
            axis=1
        )

        # Need normalized article entity vectors
        candidate_entities = (
            article_entity_embeddings[n_idx]
            .copy()
        )

        norms = np.linalg.norm(
            candidate_entities,
            axis=1,
            keepdims=True
        )

        norms[norms == 0] = 1

        candidate_entities /= norms

        entity_scores[start:end] = np.sum(
            user_entity_embeddings[u_idx]
            * candidate_entities,
            axis=1
        )

    df["text_similarity"] = text_scores
    df["entity_similarity"] = entity_scores

    return df

In [61]:
candidate_df = add_similarity_features(
    candidate_df
)

In [62]:
print(
    candidate_df[
        ["text_similarity", "entity_similarity"]
    ].describe()
)

assert candidate_df[
    "text_similarity"
].between(-1.001, 1.001).all()

assert candidate_df[
    "entity_similarity"
].between(-1.001, 1.001).all()

       text_similarity  entity_similarity
count     5.843444e+06       5.843444e+06
mean      1.904230e-01       3.345374e-01
std       1.243969e-01       2.784663e-01
min      -2.381785e-01      -6.091755e-01
25%       1.007199e-01       0.000000e+00
50%       1.826399e-01       3.370971e-01
75%       2.714400e-01       5.635023e-01
max       1.000000e+00       1.000000e+00


Entity overlap ratio

How many of the entities in this candidate news article have already appeared in this user’s clicked history

In [63]:
user_entity_sets = []

for history in beh["history_ids"]:

    entities = set()

    for news_id in history:
        entities.update(
            entity_list_lookup.get(
                news_id,
                []
            )
        )

    user_entity_sets.append(entities)

def calculate_entity_overlap(
    behavior_row_id,
    news_id
):
    candidate_entities = set(
        entity_list_lookup.get(
            news_id,
            []
        )
    )

    if len(candidate_entities) == 0:
        return 0.0

    user_entities = user_entity_sets[
        behavior_row_id
    ]

    return (
        len(
            candidate_entities
            & user_entities
        )
        / len(candidate_entities)
    )

In [71]:
candidate_df["entity_overlap_ratio"] = np.fromiter(
    (
        calculate_entity_overlap(behavior_row_id, news_id)
        for behavior_row_id, news_id in zip(
            candidate_df["behavior_row_id"],
            candidate_df["news_id"]
        )
    ),
    dtype=np.float32,
    count=len(candidate_df)
)

assert candidate_df["entity_overlap_ratio"].between(0, 1).all()

candidate_df[
    ["news_id", "entity_overlap_ratio"]
].head()

,news_id,entity_overlap_ratio
0,N55689,0.000000
1,N35729,0.000000
2,N20678,0.000000
3,N39317,0.333333
4,N58114,0.000000


In [64]:
print(len(candidate_df))

print(candidate_df["text_similarity"].notna().sum())
print(candidate_df["entity_similarity"].notna().sum())

5843444
5843444
5843444


Semantic neighbour CTR

In [73]:
from sentence_transformers import util
import torch

embedding_tensor = torch.tensor(
    article_text_embeddings,
    dtype=torch.float32
)

#Use top_k=11 since the closest article found will be itself, so to find the next 10 closest need to use top_k=11
semantic_hits = util.semantic_search(
    embedding_tensor,
    embedding_tensor,
    top_k=11,
    query_chunk_size=256,
    corpus_chunk_size=5000
)

In [66]:
semantic_neighbors = {}

news_ids = news_df["news_id"].tolist()

for i, hits in enumerate(semantic_hits):

    current_news = news_ids[i]

    neighbors = []

    for hit in hits:

        neighbor_news = news_ids[
            hit["corpus_id"]
        ]

        if neighbor_news == current_news:
            continue

        similarity = max(
            float(hit["score"]),
            0
        )

        neighbors.append(
            (
                neighbor_news,
                similarity
            )
        )

        if len(neighbors) == 10:
            break

    semantic_neighbors[current_news] = neighbors

In [74]:
example_news = news_ids[0]

print(example_news)
print(semantic_neighbors[example_news])

assert all(
    n != example_news
    for n, score in semantic_neighbors[example_news]
)

N55528
[('N42227', 0.6399319171905518), ('N55244', 0.5964511036872864), ('N57591', 0.5949617028236389), ('N23273', 0.5869383811950684), ('N43522', 0.5823887586593628), ('N61541', 0.5811440944671631), ('N7898', 0.5777686834335327), ('N23937', 0.5757788419723511), ('N19211', 0.5720764994621277), ('N1810', 0.5719085931777954)]


In [75]:
semantic_neighbors["N123"]

[('N65132', 0.4507255256175995),
 ('N51636', 0.4363856911659241),
 ('N21608', 0.43261173367500305),
 ('N43000', 0.43001219630241394),
 ('N32339', 0.4238063395023346),
 ('N36561', 0.411257266998291),
 ('N65111', 0.4091908037662506),
 ('N1050', 0.40772905945777893),
 ('N43865', 0.4049615263938904),
 ('N17641', 0.40449419617652893)]

Content smoothed CTR

In [68]:
content_ctr_features = [
    "category_recent_ctr_6h",
    "subcategory_recent_ctr_6h",
    "entity_recent_ctr_mean_6h"
]

#Find average  of CTR of category, subcategory and entity
candidate_df["content_prior_ctr"] = (
    candidate_df[
        content_ctr_features
    ]
    .mean(
        axis=1,
        skipna=True
    )
)

ALPHA = 20

candidate_df["content_smoothed_ctr"] = (
    (
        candidate_df["prior_clicks"]
        + ALPHA
        * candidate_df["content_prior_ctr"]
    )
    /
    (
        candidate_df["prior_exposures"]
        + ALPHA
    )
)

assert candidate_df[
    "content_prior_ctr"
].dropna().between(0, 1).all()

assert candidate_df[
    "content_smoothed_ctr"
].dropna().between(0, 1).all()

In [87]:
#Check cold examples
train_model_df[
    [
        "news_id",
        "prior_exposures",
        "prior_clicks",
        "prior_ctr",
        "content_prior_ctr",
        "content_smoothed_ctr"
    ]
].dropna().sort_values(
    "prior_exposures"
).head(20)

,news_id,prior_exposures,prior_clicks,prior_ctr,content_prior_ctr,content_smoothed_ctr
61154,N9738,1,0,0.0,0.033827,0.032216
491086,N16442,1,0,0.0,0.043987,0.041892
61838,N64898,1,0,0.0,0.048503,0.046193
228063,N59967,1,0,0.0,0.080270,0.076448
605550,N634,1,0,0.0,0.080994,0.077137
294162,N25443,1,0,0.0,0.037482,0.035697
597691,N20626,1,0,0.0,0.047369,0.045113
99518,N9213,1,0,0.0,0.027532,0.026221
505756,N19377,1,0,0.0,0.025831,0.024601
328456,N30172,1,0,0.0,0.021759,0.020722


In [86]:
#Check warm examples
train_model_df[
    [
        "news_id",
        "prior_exposures",
        "prior_clicks",
        "prior_ctr",
        "content_prior_ctr",
        "content_smoothed_ctr"
    ]
].dropna().sort_values(
    "prior_exposures",
    ascending=False
).head(20)

,news_id,prior_exposures,prior_clicks,prior_ctr,content_prior_ctr,content_smoothed_ctr
76111,N55689,18304,4315,0.235741,0.072141,0.235562
229302,N55689,18294,4314,0.235815,0.050760,0.235613
347995,N55689,18292,4313,0.235786,0.051268,0.235585
576591,N55689,18290,4313,0.235812,0.052650,0.235612
27048,N55689,18288,4313,0.235838,0.053504,0.235639
387117,N55689,18286,4312,0.235809,0.052964,0.235609
31590,N55689,18283,4312,0.235848,0.053200,0.235648
398090,N55689,18282,4311,0.235806,0.053596,0.235607
566877,N55689,18275,4310,0.235841,0.051738,0.235640
132600,N55689,18273,4310,0.235867,0.049520,0.235663


Train - Validation - Test Split

In [76]:
train_df = candidate_df[
    candidate_df["time"] < "2019-11-13"
].copy()

val_df = candidate_df[
    (candidate_df["time"] >= "2019-11-13")
    & (candidate_df["time"] < "2019-11-14")
].copy()

test_df = candidate_df[
    candidate_df["time"] >= "2019-11-14"
].copy()

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

Train: 3389342
Validation: 1231673
Test: 1222429


In [77]:
train_df["label"].value_counts()

label
0    3247784
1     141558
Name: count, dtype: int64

In [78]:
def sample_negatives(df, negatives_per_positive=4):

    positives = df[df["label"] == 1]
    negatives = df[df["label"] == 0]

    sampled_parts = []

    for impression_id, pos_group in positives.groupby("impression_id"):

        neg_group = negatives[
            negatives["impression_id"] == impression_id
        ]

        n = min(
            len(neg_group),
            negatives_per_positive * len(pos_group)
        )

        sampled_neg = neg_group.sample(
            n=n,
            random_state=42
        )

        sampled_parts.append(pos_group)
        sampled_parts.append(sampled_neg)

    return pd.concat(
        sampled_parts,
        ignore_index=True
    )

In [79]:
train_model_df = sample_negatives(train_df)

val_model_df = val_df.copy()
test_model_df = test_df.copy()

Calculate remaining missing semantic neighbour ctr

In [80]:
article_history = {}

for news_id, group in article_time.groupby("news_id"):

    group = group.sort_values("time")

    times = group["time"].to_numpy(
        dtype="datetime64[ns]"
    )

    # Total behaviour up to and including each timestamp
    cumulative_exposures = (
        group["prior_exposures"]
        + group["exposures_at_time"]
    ).to_numpy()

    cumulative_clicks = (
        group["prior_clicks"]
        + group["clicks_at_time"]
    ).to_numpy()

    article_history[news_id] = (
        times,
        cumulative_exposures,
        cumulative_clicks
    )

In [81]:
def add_semantic_neighbor_ctr(df):

    df = df.reset_index(drop=True).copy()

    scores = np.full(
        len(df),
        np.nan,
        dtype=np.float32
    )

    groups = df.groupby(
        "news_id",
        sort=False
    ).indices

    for news_id, row_indices in groups.items():

        query_times = df.loc[
            row_indices,
            "time"
        ].to_numpy(dtype="datetime64[ns]")

        weighted_clicks = np.zeros(
            len(row_indices)
        )

        weighted_exposures = np.zeros(
            len(row_indices)
        )

        for neighbor_id, similarity in semantic_neighbors.get(
            news_id, []
        ):

            if neighbor_id not in article_history:
                continue

            times, exposures, clicks = (
                article_history[neighbor_id]
            )

            # Find latest neighbour behaviour STRICTLY before
            # the candidate's prediction time
            positions = np.searchsorted(
                times,
                query_times,
                side="left"
            ) - 1

            valid = positions >= 0

            weighted_clicks[valid] += (
                similarity
                * clicks[positions[valid]]
            )

            weighted_exposures[valid] += (
                similarity
                * exposures[positions[valid]]
            )

        valid = weighted_exposures > 0

        group_scores = np.full(
            len(row_indices),
            np.nan
        )

        group_scores[valid] = (
            weighted_clicks[valid]
            / weighted_exposures[valid]
        )

        scores[row_indices] = group_scores

    df["semantic_neighbor_ctr"] = scores

    return df

In [82]:
train_model_df = add_semantic_neighbor_ctr(
    train_model_df
)

val_model_df = add_semantic_neighbor_ctr(
    val_model_df
)

test_model_df = add_semantic_neighbor_ctr(
    test_model_df
)

In [83]:
#Verify if successfully created

for df in [
    train_model_df,
    val_model_df,
    test_model_df
]:
    assert df[
        "semantic_neighbor_ctr"
    ].dropna().between(0, 1).all()

print(
    train_model_df[
        "semantic_neighbor_ctr"
    ].describe()
)

count    563742.000000
mean          0.046511
std           0.067212
min           0.000000
25%           0.005932
50%           0.028009
75%           0.064266
max           1.000000
Name: semantic_neighbor_ctr, dtype: float64


In [84]:
content_ctr_features = [
    "category_recent_ctr_6h",
    "subcategory_recent_ctr_6h",
    "entity_recent_ctr_mean_6h",
    "semantic_neighbor_ctr"
]

ALPHA = 20

for df in [
    train_model_df,
    val_model_df,
    test_model_df
]:

    df["content_prior_ctr"] = (
        df[content_ctr_features]
        .mean(
            axis=1,
            skipna=True
        )
    )

    df["content_smoothed_ctr"] = (
        df["prior_clicks"]
        + ALPHA * df["content_prior_ctr"]
    ) / (
        df["prior_exposures"]
        + ALPHA
    )

Evaluation Labels, NOT TO BE USED FOR TRAINING

In [85]:
for df in [
    train_model_df,
    val_model_df,
    test_model_df
]:

    df["cold_flag"] = (
        df["prior_exposures"] < 5
    )

    df["zero_prior_click_flag"] = (
        df["prior_clicks"] == 0
    )

    df["exposure_bucket"] = pd.cut(
        df["prior_exposures"],
        bins=[
            -1,
            0,
            4,
            20,
            100,
            np.inf
        ],
        labels=[
            "0",
            "1-4",
            "5-20",
            "21-100",
            ">100"
        ]
    )

In [94]:
import os
import numpy as np

os.makedirs("processed_data", exist_ok=True)

train_model_df.to_pickle(
    "processed_data/train_model_df.pkl"
)

val_model_df.to_pickle(
    "processed_data/val_model_df.pkl"
)

test_model_df.to_pickle(
    "processed_data/test_model_df.pkl"
)

np.save(
    "processed_data/article_text_embeddings.npy",
    article_text_embeddings
)

np.save(
    "processed_data/article_entity_embeddings.npy",
    article_entity_embeddings
)

news_df[["news_id"]].to_csv(
    "processed_data/news_embedding_index.csv",
    index=False
)

Final files to be used:

train_model_df.pkl
val_model_df.pkl
test_model_df.pkl

In [96]:
feature_cols = [
    "article_age_hours",
    "prior_exposures",
    "prior_clicks",
    "prior_ctr",
    "behavior_confidence",
    "recent_exposures_6h",
    "recent_clicks_6h",
    "recent_ctr_6h",
    "ctr_momentum_6h",
    "category_recent_ctr_6h",
    "category_ctr_momentum_6h",
    "subcategory_recent_ctr_6h",
    "subcategory_ctr_momentum_6h",
    "entity_recent_ctr_mean_6h",
    "entity_ctr_momentum_mean_6h",
    "entity_ctr_momentum_max_6h",
    "semantic_neighbor_ctr",
    "content_prior_ctr",
    "content_smoothed_ctr",
    "history_length",
    "category_affinity",
    "subcategory_affinity",
    "text_similarity",
    "entity_similarity",
    "entity_overlap_ratio",
]

In [97]:
x_train = train_model_df[feature_cols]
y_train = train_model_df["label"]

x_val = val_model_df[feature_cols]
y_val = val_model_df["label"]

x_test = test_model_df[feature_cols]
y_test = test_model_df["label"]